In [ ]:
import numpy as np
from generate_3d_mesh import toroidal_mesh, stretched_toroidal_mesh, visualize_mesh, spherical_mesh
from parametric_tube import sigmoid_curve, spiral_curve, loop_with_twist, custom_curve_tube_mesh
from radius_functions import *

In [ ]:
LON = 30
LAT = 45

## Basic Toroidal Structure

In [ ]:
donut_vertices, donut_triangles = toroidal_mesh(LON, LAT)
print(f"Mesh created with {len(donut_vertices)} vertices and {len(donut_triangles)} triangles")

In [ ]:
fig = visualize_mesh(donut_vertices, donut_triangles, "Solar Prominence Loop",
                    renderer='notebook',
                    return_fig=True)
fig.update_layout(template='plotly_dark', showlegend=False)
fig.show()

## Stretched Toroidal Structure

In [ ]:
# Create an elongated torus that stretches along the x-axis
stretched_donut_vertices, stretched_donut_triangles = stretched_toroidal_mesh(
    LON, LAT,                 # longitude, latitude on sun
    stretch_factor=2.0,     # How much to stretch (2.0 = twice as long)
    stretch_axis='x',       # Stretch along x-axis
    asymmetry=0.2           # Slight asymmetry for more natural look
)
print(f"Mesh created with {len(stretched_donut_vertices)} vertices and {len(stretched_donut_triangles)} triangles")

In [ ]:
# Visualize the torus
fig = visualize_mesh(stretched_donut_vertices, stretched_donut_triangles, "Stretched Solar Prominence",
                    renderer='notebook',
                    return_fig=True)
fig.update_layout(template='plotly_dark', showlegend=False)
fig.show()


## Custom Tube Curves

Toroidal structures cn be modified with the `custom_curve_tube_mesh` function.
This following arguments can be passed:

    Args:
        lon0_deg, lat0_deg:     Position on sun surface (degrees)
        R_sun:                  Radius of sun
        tube_radius:            Radius of the tube
        n_curve:                Resolution along curve
        n_circle:               Resolution around tube circumference
        curve_func:             Function that takes parameter t (0 to 1) and returns (x,y,z)
                                If None, defaults to a simple arch curve
        curve_params:           Dictionary of parameters to pass to curve_func
        radius_func:            Function that takes parameter t (0 to 1) and returns radius multiplier

There are multiple pre-defined curve functions:
* `sigmoid_curve`   (_t_, height=2.0, width=3.0, asymmetry=0.0, turns=1.0)
* `spiral_curve`    (_t_, height=2.0, radius=2.0, turns=2.5, taper=0.3)
* `loop_with_twist` (_t_, height=2.0, width=3.0, twist=1.5)

For further information, check out 'parametric_tube.py'.

Similarly, there are multiple pre-defined radius functions (inside 'radius_functions.py')
* `tapered_ends_radius` (_t_, taper_factor=0.5, taper_exp=2.0)
* `bulging_radius`      (_t_, bulge_radius=1.5, bulge_width=0.3)
* `multi_bulge_radius`  (_t_, n_bulges=3, bulge_factor=1.3)
* `wavy_radius`         (_t_, wavelength=8, amplitude=0.3)
* `magnetic_flux_radius`    (_t_, expansion_factor=3.0, corona_start=0.2)
* `kink_instability_radius` (_t_, kink_center=0.5, kink_width=0.1, kink_factor=1.7)
* `turbulent_radius`        (_t_, seed=42, n_points=10, amplitude=0.2)
* `composite_radius`        (_t_, radius_funcs=None, combination_method='multiply', weights=None, **kwargs)

The `composite_radius` functions takes as argument a list of radius functions that can be combined through 'multiply', 'max', 'add', or 'weighted'. For the 'weighted' method, a list of weights for each function can be passed. Keyword arguments take the function name as key with a dictionary of arguments for that function as values.

For further information, check out 'radius_functions.py'.





### Default Arch

In [ ]:
# Example 1: Default arch curve
parametric_vertices, parametric_triangles = custom_curve_tube_mesh(
    sun_position=(0, 0),
    orientation=(0, 0),
    R_sun=10.0,         # Sun radius
    base_radius=0.5,    # Tube thickness
    n_curve=40,         # Resolution along curve
    n_circle=15         # Resolution around tube
)
print(f"Mesh created with {len(parametric_vertices)} vertices and {len(parametric_triangles)} triangles")

In [ ]:
# Visualize the arch
fig = visualize_mesh(parametric_vertices, parametric_triangles, "Simple Arch Prominence",
                   renderer='notebook',
                   return_fig=True)
fig.update_layout(template='plotly_dark', showlegend=False)
fig.show()

### Twisted Loop Prominence

In [ ]:
# Example 4: Twisted loop
vertices_loop, triangles_loop = custom_curve_tube_mesh(
    sun_position=(90, -45),
    orientation=(0, 0),               # Yet another position
    base_radius=0.35,   # Thicker tube
    curve_func=loop_with_twist,
    curve_params={'height': 2.2, 'width': 8, 'twist': 1.1},
    radius_func=lambda t: magnetic_flux_radius(t, expansion_factor=2.0)
)
print(f"Mesh created with {len(vertices_loop)} vertices and {len(triangles_loop)} triangles")

In [ ]:
# Visualize the twisted loop
fig = visualize_mesh(vertices_loop, triangles_loop, "Twisted Loop Prominence",
                   renderer='notebook',
                   return_fig=True)
fig.update_layout(template='plotly_dark', showlegend=False)
fig.show()


### Twisted Loop Kink Instability

In [ ]:
# Example 4: Twisted loop
vertices_loop, triangles_loop = custom_curve_tube_mesh(
    sun_position=(90, -45),
    orientation=(0, 0),
    base_radius=0.35,   # Thicker tube
    curve_func=loop_with_twist,
    curve_params={'height': 2.2, 'width': 8, 'twist': 1.1},
    radius_func=lambda t: kink_instability_radius(t, kink_center=.5, kink_width=.1, kink_factor=2.0)
)
print(f"Mesh created with {len(vertices_loop)} vertices and {len(triangles_loop)} triangles")

In [ ]:
# Visualize the twisted loop
fig = visualize_mesh(vertices_loop, triangles_loop, "Twisted Loop Prominence",
                   renderer='notebook',
                   return_fig=True)
fig.update_layout(template='plotly_dark', showlegend=False)
fig.show()

### Combination of Multiple Radius Functions

In [ ]:
# Directly with specified functions
vertices_composite, triangles_composite = custom_curve_tube_mesh(
    sun_position=(10, -60),           # Longitude, latitude
    orientation=(0, 0),               # Orientation w.r.t surface normal
    base_radius=0.3,                  # Base thickness
    curve_func=loop_with_twist,       # Curve shape
    curve_params={'height': 2.2, 'width': 8, 'twist': 1.1},
    radius_func=lambda t: composite_radius(
        t,
        radius_funcs=[magnetic_flux_radius, wavy_radius, turbulent_radius],
        combination_method='add',
        # Parameters for each function
        magnetic_flux_radius={'expansion_factor': 2.5, 'corona_start': .5},
        wavy_radius={'wavelength': 10, 'amplitude': 0.2},
        kink_instability_radius={'kink_center': 0.7, 'kink_factor': 2},
        turbulent_radius={'amplitude': 0.3, 'seed': 123}
    )
)

print(f"Mesh created with {len(vertices_composite)} vertices and {len(triangles_composite)} triangles")

In [ ]:
# Visualize the twisted loop
fig = visualize_mesh(vertices_composite, triangles_composite, "Twisted Loop Prominence",
                   renderer='notebook',
                   return_fig=True)
fig.update_layout(template='plotly_dark', showlegend=False)
fig.show()

In [ ]:
# Example 4: Twisted loop
vertices_sigmoid, triangles_sigmoid = custom_curve_tube_mesh(
    sun_position=(90, -45),
    orientation=(0, 0),
    base_radius=0.35,  # Thicker tube
    curve_func=sigmoid_curve,
    curve_params={'height': 2.2, 'width': 8, 'asymmetry': .5, 'turns': 1.0},
    radius_func=lambda t: magnetic_flux_radius(t, expansion_factor=2.0, corona_start=2)
)
print(f"Mesh created with {len(vertices_sigmoid)} vertices and {len(triangles_sigmoid)} triangles")

In [ ]:
# Visualize the sigmoid
fig = visualize_mesh(vertices_sigmoid, triangles_sigmoid, "Sigmoid",
                     renderer='notebook',
                     return_fig=True)
fig.update_layout(template='plotly_dark', showlegend=False)
fig.show()

In [ ]:
def combine_meshes(sphere_vertices, sphere_triangles, tube_meshes):
    """
    Combine a spherical mesh with multiple tube structures (like solar prominences).

    Args:
        sphere_vertices: Vertices of the spherical mesh
        sphere_triangles: Triangle indices of the spherical mesh
        tube_meshes: List of tuples, each containing (vertices, triangles) for a tube
        show_anchors: Whether to show anchor points for debugging

    Returns:
        tuple: (combined_vertices, combined_triangles)
    """

    # Start with the sphere vertices and triangles
    combined_vertices = sphere_vertices.copy()
    combined_triangles = sphere_triangles.copy()

    # For each tube mesh
    for i, (tube_vertices, tube_triangles) in enumerate(tube_meshes):
        # Get the current vertex count to offset triangle indices
        vertex_offset = len(combined_vertices)

        # Add the tube vertices to the combined vertices
        combined_vertices = np.vstack([combined_vertices, tube_vertices])

        # Add the tube triangles with adjusted indices
        adjusted_triangles = tube_triangles + vertex_offset
        combined_triangles = np.vstack([combined_triangles, adjusted_triangles])

    return combined_vertices, combined_triangles


In [ ]:
# Create a grid of longitude and latitude points
lon_steps, lat_steps = 15, 10
longitudes = np.linspace(-180, 180, lon_steps)
latitudes = np.linspace(-90, 90, lat_steps)

# Create a meshgrid for all combinations
lon_grid, lat_grid = np.meshgrid(longitudes, latitudes)

# Flatten the grid
lons = lon_grid.flatten()
lats = lat_grid.flatten()

# Create distances with a simple pattern (a bumpy sphere)
base_radius = 10
distances = base_radius + 1 * np.sin(np.radians(lons)) * np.cos(np.radians(lats))

# Combine into points array
sun_points = np.column_stack((lons, lats, distances))

# Generate mesh
vertices_sphere, triangles_sphere = spherical_mesh(sun_points)

In [ ]:
vertices_loop, triangles_loop = custom_curve_tube_mesh(
    sun_position=(90, -45),
    orientation=(0, 0),       # Tuple (tilt_angle, rotation_angle, roll_angle)
    base_radius=0.15,   # Thicker tube
    curve_func=loop_with_twist,
    curve_params={'height': .4, 'width': 4, 'twist': 2},
    radius_func=lambda t: kink_instability_radius(t, kink_center=.5, kink_width=.1, kink_factor=2.0)
)

vertices_loop2, triangles_loop2 = custom_curve_tube_mesh(
    sun_position=(120, 45),
    orientation=(0, 0),       # Tuple (tilt_angle, rotation_angle, roll_angle)
    base_radius=0.15,   # Thicker tube
    curve_func=loop_with_twist,
    curve_params={'height': 1, 'width': 6, 'twist': 1.1},
    radius_func=lambda t: kink_instability_radius(t, kink_center=.5, kink_width=.1, kink_factor=1.0)
)

vertices_magflux, triangles_magflux = custom_curve_tube_mesh(
    sun_position=(240, -70),
    orientation=(90, 0),       # Tuple (tilt_angle, rotation_angle, roll_angle)
    base_radius=0.05,   # Thicker tube
    curve_func=spiral_curve,
    curve_params={'height': 4.0, 'radius': .2, 'turns': 2, 'taper': 1.1},
    radius_func=lambda t: magnetic_flux_radius(t, expansion_factor=4.0, corona_start=0.2)
)

In [ ]:
vertices_total, triangles_total = combine_meshes(
    vertices_sphere, triangles_sphere,
    [
        (vertices_loop, triangles_loop),
        (vertices_loop2, triangles_loop2),
        (vertices_composite, triangles_composite),
        (vertices_magflux, triangles_magflux)
        #(vertices_sigmoid, triangles_sigmoid, {'scale': 1.2}),
        #(vertices_twisted, triangles_twisted, {'rotation': 45})
    ],
)

print(f"Mesh created with {len(vertices_total)} vertices and {len(triangles_total)} triangles")

In [ ]:
# Visualize the sphere with features
fig = visualize_mesh(
    vertices_total, triangles_total, "Fake Sun",
    renderer='notebook',
    return_fig=True)
fig.update_layout(template='plotly_dark', showlegend=False)
fig.show()